<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/standards/standards_based_engineering_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Standards-Based Engineering Checks

Use simple public screening checks to flag line velocity, pressure drop, and relief-load issues before a formal standards review.


## Setup

Install imports and define the gas fluid.


In [ ]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from neqsim import jneqsim
except ImportError:
    from neqsim import jNeqSim as jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

def make_gas(temperature_c=25.0, pressure_bara=60.0):
    fluid = SystemSrkEos(273.15 + temperature_c, pressure_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.86)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.03)
    fluid.addComponent("n-butane", 0.01)
    fluid.setMixingRule("classic")
    TPflash(fluid)
    fluid.initProperties()
    return fluid

## Define Design Cases

These inputs represent early design-screening values.


In [ ]:
cases = pd.DataFrame({
    "line": ["A", "B", "C"],
    "flow_kg_per_hr": [5000, 12000, 20000],
    "diameter_m": [0.15, 0.20, 0.25],
    "pressure_bara": [60, 80, 100],
    "temperature_C": [20, 25, 30],
})
cases


## Run Public Screening Checks

The limits are educational placeholders and must not replace project-specific standards interpretation.


In [ ]:
import math
rows = []
for _, row in cases.iterrows():
    fluid = make_gas(float(row.temperature_C), float(row.pressure_bara))
    density = fluid.getDensity("kg/m3")
    area = math.pi * (row.diameter_m ** 2) / 4.0
    velocity = row.flow_kg_per_hr / 3600.0 / density / area
    pressure_gradient_bar_per_km = 0.02 * velocity ** 2
    rows.append({
        "line": row.line,
        "velocity_m_per_s": velocity,
        "velocity_flag": velocity > 25.0,
        "pressure_gradient_bar_per_km": pressure_gradient_bar_per_km,
        "pressure_drop_flag": pressure_gradient_bar_per_km > 0.5,
    })
checks = pd.DataFrame(rows)
checks


## Summarize Findings

A flag means the case should be reviewed with the governing project standard and a validated calculation method.


In [ ]:
checks.assign(any_flag=checks[["velocity_flag", "pressure_drop_flag"]].any(axis=1))
